## Import Libraries

In [24]:
import numpy as np
import pandas as pd

## Load the Dataset

In [25]:
df = pd.read_csv('../data/processed/rainfall_features.csv')

In [26]:
df.head()

,SUBDIVISION,YEAR,JAN,FEB,MAR,APR,MAY,JUN,JUL,AUG,...,Jun-Sep,Oct-Dec,annual_lag1,annual_lag2,annual_lag3,rolling_mean_3,annual_lag4,annual_lag5,rolling_mean_5,target
0,ANDAMAN & NICOBAR ISLANDS,1901,49.2,87.1,29.2,2.3,528.8,517.5,365.1,481.1,...,1696.3,980.3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3520.7
1,ANDAMAN & NICOBAR ISLANDS,1902,0.0,159.8,12.2,0.0,446.1,537.1,228.9,753.7,...,2185.9,716.7,3373.2,NaN,NaN,NaN,NaN,NaN,NaN,2957.4
2,ANDAMAN & NICOBAR ISLANDS,1903,12.7,144.0,0.0,1.0,235.1,479.9,728.4,326.7,...,1874.0,690.6,3520.7,3373.2,NaN,NaN,NaN,NaN,NaN,3079.6
3,ANDAMAN & NICOBAR ISLANDS,1904,9.4,14.7,0.0,202.4,304.5,495.1,502.0,160.1,...,1977.6,571.0,2957.4,3520.7,3373.2,3283.766667,NaN,NaN,NaN,2566.7
4,ANDAMAN & NICOBAR ISLANDS,1905,1.3,0.0,3.3,26.9,279.5,628.7,368.7,330.5,...,1624.9,630.8,3079.6,2957.4,3520.7,3185.900000,3373.2,NaN,NaN,2534.4


In [27]:
df.shape

(4116, 27)

In [28]:
df.columns

Index(['SUBDIVISION', 'YEAR', 'JAN', 'FEB', 'MAR', 'APR', 'MAY', 'JUN', 'JUL',
       'AUG', 'SEP', 'OCT', 'NOV', 'DEC', 'ANNUAL', 'Jan-Feb', 'Mar-May',
       'Jun-Sep', 'Oct-Dec', 'annual_lag1', 'annual_lag2', 'annual_lag3',
       'rolling_mean_3', 'annual_lag4', 'annual_lag5', 'rolling_mean_5',
       'target'],
      dtype='object')

In [29]:
print(df.columns.tolist())

['SUBDIVISION', 'YEAR', 'JAN', 'FEB', 'MAR', 'APR', 'MAY', 'JUN', 'JUL', 'AUG', 'SEP', 'OCT', 'NOV', 'DEC', 'ANNUAL', 'Jan-Feb', 'Mar-May', 'Jun-Sep', 'Oct-Dec', 'annual_lag1', 'annual_lag2', 'annual_lag3', 'rolling_mean_3', 'annual_lag4', 'annual_lag5', 'rolling_mean_5', 'target']


In [30]:
df.isnull().sum()

SUBDIVISION         0
YEAR                0
JAN                 0
FEB                 0
MAR                 0
APR                 0
MAY                 0
JUN                 0
JUL                 0
AUG                 0
SEP                 0
OCT                 0
NOV                 0
DEC                 0
ANNUAL              0
Jan-Feb             0
Mar-May             0
Jun-Sep             0
Oct-Dec             0
annual_lag1        41
annual_lag2        79
annual_lag3       116
rolling_mean_3    122
annual_lag4       152
annual_lag5       188
rolling_mean_5    202
target             41
dtype: int64

In [31]:
df.duplicated().sum()

np.int64(0)

In [32]:
duplicate_keys = df.duplicated(
    subset=['SUBDIVISION', 'YEAR']
).sum()

print("Duplicate SUBDIVISION-YEAR combinations:", duplicate_keys)

Duplicate SUBDIVISION-YEAR combinations: 0


#### Check feature and target alignment

In [33]:
df[
    [
        'SUBDIVISION',
        'YEAR',
        'annual_lag1',
        'annual_lag2',
        'annual_lag3',
        'rolling_mean_3',
        'target'
    ]
].head(10)

,SUBDIVISION,YEAR,annual_lag1,annual_lag2,annual_lag3,rolling_mean_3,target
0,ANDAMAN & NICOBAR ISLANDS,1901,NaN,NaN,NaN,NaN,3520.7
1,ANDAMAN & NICOBAR ISLANDS,1902,3373.2,NaN,NaN,NaN,2957.4
2,ANDAMAN & NICOBAR ISLANDS,1903,3520.7,3373.2,NaN,NaN,3079.6
3,ANDAMAN & NICOBAR ISLANDS,1904,2957.4,3520.7,3373.2,3283.766667,2566.7
4,ANDAMAN & NICOBAR ISLANDS,1905,3079.6,2957.4,3520.7,3185.900000,2534.4
5,ANDAMAN & NICOBAR ISLANDS,1906,2566.7,3079.6,2957.4,2867.900000,3347.9
6,ANDAMAN & NICOBAR ISLANDS,1907,2534.4,2566.7,3079.6,2726.900000,3576.4
7,ANDAMAN & NICOBAR ISLANDS,1908,3347.9,2534.4,2566.7,2816.333333,NaN
8,ANDAMAN & NICOBAR ISLANDS,1910,NaN,3576.4,3347.9,NaN,2687.2
9,ANDAMAN & NICOBAR ISLANDS,1911,2899.4,NaN,3576.4,NaN,2960.5


In [34]:
df[
    (df['SUBDIVISION'] == 'ANDAMAN & NICOBAR ISLANDS') &
    (df['YEAR'].between(1907, 1911))
][
    [
        'SUBDIVISION',
        'YEAR',
        'annual_lag1',
        'annual_lag2',
        'annual_lag3',
        'rolling_mean_3',
        'target'
    ]
]

,SUBDIVISION,YEAR,annual_lag1,annual_lag2,annual_lag3,rolling_mean_3,target
6,ANDAMAN & NICOBAR ISLANDS,1907,2534.4,2566.7,3079.6,2726.900000,3576.4
7,ANDAMAN & NICOBAR ISLANDS,1908,3347.9,2534.4,2566.7,2816.333333,NaN
8,ANDAMAN & NICOBAR ISLANDS,1910,NaN,3576.4,3347.9,NaN,2687.2
9,ANDAMAN & NICOBAR ISLANDS,1911,2899.4,NaN,3576.4,NaN,2960.5


In [37]:
# Check whether target is correctly aligned with the next calendar year
annual_lookup = df.set_index(
    ['SUBDIVISION', 'YEAR']
)['ANNUAL']
target_check = []

for subdivision, year in zip(df['SUBDIVISION'], df['YEAR']):
    expected_target = annual_lookup.get(
        (subdivision, year + 1),
        np.nan
    )
    target_check.append(expected_target)

comparison = pd.DataFrame({
    'actual_target': df['target'],
    'expected_target': target_check
})

target_match = (
    comparison['actual_target'].eq(comparison['expected_target'])
    | (
        comparison['actual_target'].isna()
        & comparison['expected_target'].isna()
    )
)

print("Target mismatches:", (~target_match).sum())

Target mismatches: 0


#### Check infinite values

In [38]:
infinite_values = np.isinf(
    df.select_dtypes(include='number')
).sum().sum()

print("Infinite values:", infinite_values)

Infinite values: 0


#### Check feature ranges

In [39]:
feature_cols = [
    'annual_lag1',
    'annual_lag2',
    'annual_lag3',
    'annual_lag4',
    'annual_lag5',
    'rolling_mean_3',
    'rolling_mean_5',
    'target'
]

df[feature_cols].describe()

,annual_lag1,annual_lag2,annual_lag3,annual_lag4,annual_lag5,rolling_mean_3,rolling_mean_5,target
count,4075.000000,4037.000000,4000.000000,3964.000000,3928.000000,3994.000000,3914.000000,4075.000000
mean,1416.973242,1418.157034,1417.272712,1418.207501,1417.715385,1415.081504,1412.567249,1416.051748
std,908.349967,909.137739,909.009704,908.951561,906.957848,881.731300,872.919594,906.583830
min,62.200000,62.200000,62.200000,62.200000,62.200000,181.500000,188.760000,62.200000
25%,806.450000,807.200000,806.475000,807.475000,807.750000,823.000000,827.595000,806.450000
50%,1124.600000,1126.100000,1124.650000,1126.350000,1126.850000,1112.733333,1100.730000,1124.600000
75%,1651.250000,1652.602784,1651.925696,1651.925696,1652.852088,1589.250000,1566.665000,1651.250000
max,6331.100000,6331.100000,6331.100000,6331.100000,6331.100000,5191.666667,4833.100000,6331.100000


#### Find how many rows are actually model-ready

In [40]:
model_features = [
    'annual_lag1',
    'annual_lag2',
    'annual_lag3',
    'annual_lag4',
    'annual_lag5',
    'rolling_mean_3',
    'rolling_mean_5',
    'target'
]

complete_rows = df[model_features].notna().all(axis=1)

print("Total rows:", len(df))
print("Complete model-ready rows:", complete_rows.sum())
print("Rows with missing values:", (~complete_rows).sum())

Total rows: 4116
Complete model-ready rows: 3874
Rows with missing values: 242


In [41]:
missing_by_feature = df[model_features].isna().sum()

print(missing_by_feature)

annual_lag1        41
annual_lag2        79
annual_lag3       116
annual_lag4       152
annual_lag5       188
rolling_mean_3    122
rolling_mean_5    202
target             41
dtype: int64


In [42]:
incomplete_rows = df.loc[
    ~df[model_features].notna().all(axis=1)
]

incomplete_by_subdivision = (
    incomplete_rows['SUBDIVISION']
    .value_counts()
)

print(incomplete_by_subdivision)

SUBDIVISION
ANDAMAN & NICOBAR ISLANDS             20
ARUNACHAL PRADESH                     12
LAKSHADWEEP                           12
BIHAR                                  6
MATATHWADA                             6
WEST RAJASTHAN                         6
WEST MADHYA PRADESH                    6
VIDARBHA                               6
UTTARAKHAND                            6
TELANGANA                              6
TAMIL NADU                             6
SUB HIMALAYAN WEST BENGAL & SIKKIM     6
SOUTH INTERIOR KARNATAKA               6
SAURASHTRA & KUTCH                     6
RAYALSEEMA                             6
PUNJAB                                 6
ORISSA                                 6
NORTH INTERIOR KARNATAKA               6
NAGA MANI MIZO TRIPURA                 6
MADHYA MAHARASHTRA                     6
CHHATTISGARH                           6
ASSAM & MEGHALAYA                      6
KONKAN & GOA                           6
KERALA                                 6
JHAR

In [43]:
print(
    incomplete_rows[
        ['SUBDIVISION', 'YEAR']
    ].head(20)
)

                   SUBDIVISION  YEAR
0    ANDAMAN & NICOBAR ISLANDS  1901
1    ANDAMAN & NICOBAR ISLANDS  1902
2    ANDAMAN & NICOBAR ISLANDS  1903
3    ANDAMAN & NICOBAR ISLANDS  1904
4    ANDAMAN & NICOBAR ISLANDS  1905
7    ANDAMAN & NICOBAR ISLANDS  1908
8    ANDAMAN & NICOBAR ISLANDS  1910
9    ANDAMAN & NICOBAR ISLANDS  1911
10   ANDAMAN & NICOBAR ISLANDS  1912
11   ANDAMAN & NICOBAR ISLANDS  1913
12   ANDAMAN & NICOBAR ISLANDS  1914
40   ANDAMAN & NICOBAR ISLANDS  1942
41   ANDAMAN & NICOBAR ISLANDS  1946
42   ANDAMAN & NICOBAR ISLANDS  1947
43   ANDAMAN & NICOBAR ISLANDS  1949
44   ANDAMAN & NICOBAR ISLANDS  1950
45   ANDAMAN & NICOBAR ISLANDS  1951
46   ANDAMAN & NICOBAR ISLANDS  1952
47   ANDAMAN & NICOBAR ISLANDS  1953
109  ANDAMAN & NICOBAR ISLANDS  2015


In [44]:
model_ready_df = df.loc[
    df[model_features].notna().all(axis=1)
].copy()

print("Model-ready shape:", model_ready_df.shape)

Model-ready shape: (3874, 27)


In [45]:
print(
    model_ready_df[model_features].isna().sum()
)

annual_lag1       0
annual_lag2       0
annual_lag3       0
annual_lag4       0
annual_lag5       0
rolling_mean_3    0
rolling_mean_5    0
target            0
dtype: int64


In [46]:
is_sorted = (
    model_ready_df
    .sort_values(['SUBDIVISION', 'YEAR'])
    .index
    .equals(model_ready_df.index)
)

print("Data is sorted by SUBDIVISION and YEAR:", is_sorted)

Data is sorted by SUBDIVISION and YEAR: True


In [47]:
print("Minimum target:", model_ready_df['target'].min())
print("Maximum target:", model_ready_df['target'].max())
print("Negative target values:", (model_ready_df['target'] < 0).sum())

Minimum target: 62.2
Maximum target: 6331.099999999999
Negative target values: 0


In [48]:
X = model_ready_df[
    [
        'annual_lag1',
        'annual_lag2',
        'annual_lag3',
        'annual_lag4',
        'annual_lag5',
        'rolling_mean_3',
        'rolling_mean_5'
    ]
]

y = model_ready_df['target']

In [49]:
X.head()

,annual_lag1,annual_lag2,annual_lag3,annual_lag4,annual_lag5,rolling_mean_3,rolling_mean_5
5,2566.7,3079.6,2957.4,3520.7,3373.2,2867.900000,3099.52
6,2534.4,2566.7,3079.6,2957.4,3520.7,2726.900000,2931.76
13,2957.8,2365.8,2960.5,2687.2,2899.4,2761.366667,2774.14
14,2741.3,2957.8,2365.8,2960.5,2687.2,2688.300000,2742.52
15,2937.5,2741.3,2957.8,2365.8,2960.5,2878.866667,2792.58


In [50]:
print("X shape:", X.shape)
print("y shape:", y.shape)
print("X missing values:", X.isna().sum().sum())
print("y missing values:", y.isna().sum())

X shape: (3874, 7)
y shape: (3874,)
X missing values: 0
y missing values: 0


In [51]:
print("X and y have same number of rows:", len(X) == len(y))

X and y have same number of rows: True


In [52]:
model_ready_df.to_csv(
    '../data/processed/rainfall_model_ready.csv',
    index=False
)

## Feature Validation — Summary